# massive in silico screening with Prophet

This notebook demonstrates how to make predictions with Prophet with any of the checkpoints we have made available.

In [5]:

%load_ext autoreload
%autoreload 2

import pandas as pd
import yaml
from prophet import Prophet

from prophet.core.config import set_config

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Load in the same config file that was used for finetuning, for the embedding files.

In [6]:
with open("config_file_finetuning.yaml", "r") as f:
    config = set_config(yaml.safe_load(f))

In [7]:
model = Prophet(
    iv_emb_path=config.genes_prior,
    cl_emb_path=config.cell_lines_prior,
    ph_emb_path=None,
    model_pth=config.ckpt_path
)

Learning rate set to 1e-05


Suppose we have some small molecules, some cell lines we would like to test them in, and we're interested in measuring their relative IC50. We can pass in lists of these inputs, and Prophet will return predictions for all combinations:

### Making predictions by passing all treatments, cell lines, and phenotypes you want to run

This format can be useful when running large combinatorial screens in silico, as it splits the experiments up into batches to help prevent memory errors.

In [8]:
iv_list = [
    "oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc4=c(c=c(c=c4)i)f)=o",
    "cc(nc1=cc=cc(n(c2=o)c(c(c(n2c3cc3)=o)=c(n4c)nc5=cc=c(c=c5f)i)=c(c4=o)c)=c1)=o",
    "fc1=cc=c(c(f)=c1c(c2=cnc3=nc=c(c=c32)c4=cc=c(c=c4)cl)=o)ns(ccc)(=o)=o",
    "cs(=o)c",  # DMSO
]
cl_list = [
    "A375",
    "UACC62",
    "WM983B",
    "MALME3M",
    "A2058",
    "WM793",
    "HT144",
    "RPMI7951",
    "SKMEL2",
    "SKMEL1",
    "HMCB",
    "MDAMB435S",
    "WM1799",
    "LOXIMVI",
]
ph_list = ["GDSC"]

In [11]:
# To get compatible with new API create dataframe and give single df to predict
# Create dataframe with all combinatorial pairs of treatments iv1, iv2 and cell line
input_df = pd.MultiIndex.from_product(
    [
        iv_list,  # iv1
        iv_list,  # iv2 
        cl_list,  # cell_line
    ],
    names=["iv1", "iv2", "cell_line"],
)
input_df = input_df.to_frame(index=False).reset_index(drop=True)
input_df["phenotype"] = ph_list[0]  # Use the first phenotype from ph_list (GSDC)

df = model.predict(input_df, save=False)

Removing 3 such as ['cc(nc1=cc=cc(n(c2=o)c(c(c(n2c3cc3)=o)=c(n4c)nc5=cc=c(c=c5f)i)=c(c4=o)c)=c1)=o', 'fc1=cc=c(c(f)=c1c(c2=cnc3=nc=c(c=c32)c4=cc=c(c=c4)cl)=o)ns(ccc)(=o)=o', 'oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc4=c(c=c(c=c4)i)f)=o'] from ['iv1', 'iv2']. 14 rows remaining.
Removing 0 such as [] from ['cell_line']. 14 rows remaining.
Filtered out 210 rows with missing embeddings. 14 rows remaining.


/home/icb/ahmet.kaya/miniconda3/envs/prophet-new/lib/python3.10/site-packages/pytorch_lightning-2.5.4-py3.10.egg/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/ahmet.kaya/miniconda3/envs/prophet-new/lib ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14]


Predicting DataLoader 0: 100%|█████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 95.78it/s]


### Making predictions for a specific set of treatments, cell lines, and phenotypes

If we're interested in only a subset of the experimental matrix, we can also pass in a custom dataframe. (This is the recommended usage, as users understand exactly the list being predicted.)

In [12]:
# Construct a dataframe containing the experiments we want to run. In practice, the user would load
# a premade dataframe here.
input_df = pd.MultiIndex.from_product(
    [
        iv_list,
        cl_list,
    ],
    names=["iv1", "cell_line"],
)
input_df = input_df.to_frame(index=False).reset_index(drop=True)
input_df["iv2"] = "cs(=o)c"  # DMSO
input_df["phenotype"] = "GDSC"
input_df

,iv1,cell_line,iv2,phenotype
0,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A375,cs(=o)c,GDSC
1,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,UACC62,cs(=o)c,GDSC
2,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM983B,cs(=o)c,GDSC
3,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,MALME3M,cs(=o)c,GDSC
4,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A2058,cs(=o)c,GDSC
5,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM793,cs(=o)c,GDSC
6,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,HT144,cs(=o)c,GDSC
7,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,RPMI7951,cs(=o)c,GDSC
8,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,SKMEL2,cs(=o)c,GDSC
9,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,SKMEL1,cs(=o)c,GDSC


In [13]:
df = model.predict(input_df, save=False)
df

Removing 3 such as ['cc(nc1=cc=cc(n(c2=o)c(c(c(n2c3cc3)=o)=c(n4c)nc5=cc=c(c=c5f)i)=c(c4=o)c)=c1)=o', 'fc1=cc=c(c(f)=c1c(c2=cnc3=nc=c(c=c32)c4=cc=c(c=c4)cl)=o)ns(ccc)(=o)=o', 'oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc4=c(c=c(c=c4)i)f)=o'] from ['iv1', 'iv2']. 14 rows remaining.
Removing 0 such as [] from ['cell_line']. 14 rows remaining.
Filtered out 42 rows with missing embeddings. 14 rows remaining.


/home/icb/ahmet.kaya/miniconda3/envs/prophet-new/lib/python3.10/site-packages/pytorch_lightning-2.5.4-py3.10.egg/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/ahmet.kaya/miniconda3/envs/prophet-new/lib ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14]


Predicting DataLoader 0: 100%|█████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 96.06it/s]


,iv1,cell_line,iv2,phenotype,pred
0,cs(=o)c,A375,cs(=o)c,GDSC,0.291144
1,cs(=o)c,UACC62,cs(=o)c,GDSC,0.219374
2,cs(=o)c,WM983B,cs(=o)c,GDSC,0.179296
3,cs(=o)c,MALME3M,cs(=o)c,GDSC,0.201030
4,cs(=o)c,A2058,cs(=o)c,GDSC,0.189866
5,cs(=o)c,WM793,cs(=o)c,GDSC,0.297630
6,cs(=o)c,HT144,cs(=o)c,GDSC,0.232367
7,cs(=o)c,RPMI7951,cs(=o)c,GDSC,0.157316
8,cs(=o)c,SKMEL2,cs(=o)c,GDSC,0.635494
9,cs(=o)c,SKMEL1,cs(=o)c,GDSC,0.236538
